In [23]:
import numpy as np
import time

class Conv2D:
    def __init__(self, num_filters, filter_size, input_channels, stride=1, padding=0):
        #num_filters: 필터(커널) 개수
        #filter_size: 필터 한 변의 크기 (정사각형 가정, 예: 3 -> 3x3)
        #input_channels: 입력 채널 수 (RGB면 3)
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.stride = stride
        self.padding = padding

        # He 초기화하기
        scale = np.sqrt(2.0 / (filter_size * filter_size * input_channels))
        self.filters = np.random.randn(
            num_filters, input_channels, filter_size, filter_size
        ) * scale
        self.biases = np.zeros(num_filters)

        # backward 계산을 위해 forward 때 입력을 저장해둠
        self.last_input = None

    def forward(self, input):
        #input shape: (batch, channels, height, width)
        #return shape: (batch, num_filters, out_h, out_w)

        self.last_input = input
        batch, channels, h, w = input.shape
        f = self.filter_size
        s = self.stride
        p = self.padding

        if p > 0:
            input = np.pad(input, ((0,0), (0,0), (p,p), (p,p)), mode='constant')

        out_h = (h + 2*p - f) // s + 1
        out_w = (w + 2*p - f) // s + 1

        output = np.zeros((batch, self.num_filters, out_h, out_w))

        # 슬라이딩 윈도우로 convolution 연산
        for i in range(out_h):
            for j in range(out_w):
                h_start, h_end = i*s, i*s + f
                w_start, w_end = j*s, j*s + f
                window = input[:, :, h_start:h_end, w_start:w_end]  # (batch, C, f, f)

                # 각 필터마다 window와 곱해서 합산
                for k in range(self.num_filters):
                    output[:, k, i, j] = np.sum(
                        window * self.filters[k], axis=(1,2,3)
                    ) + self.biases[k]

        return output

    def backward(self, d_out, lr=0.001):
        #d_out: 다음 레이어에서 넘어온 gradient (batch, num_filters, out_h, out_w)
        #return: 이전 레이어로 넘길 gradient (d_input)

        input = self.last_input
        batch, channels, h, w = input.shape
        f = self.filter_size
        s = self.stride
        p = self.padding


        padded_input = np.pad(input, ((0,0),(0,0),(p,p),(p,p)), mode='constant') if p > 0 else input
        d_input_padded = np.zeros_like(padded_input)
        d_filters = np.zeros_like(self.filters)
        d_biases = np.zeros_like(self.biases)

        out_h, out_w = d_out.shape[2], d_out.shape[3]

        for i in range(out_h):
            for j in range(out_w):
                h_start, h_end = i*s, i*s + f
                w_start, w_end = j*s, j*s + f
                window = padded_input[:, :, h_start:h_end, w_start:w_end]

                for k in range(self.num_filters):
                    # 필터 gradient: 입력 window * 해당 위치의 출력 gradient
                    d_filters[k] += np.sum(
                        window * d_out[:, k, i, j][:, None, None, None], axis=0
                    )
                    # 입력 gradient: 필터 * 출력 gradient (뒤집힌 convolution과 동치)
                    d_input_padded[:, :, h_start:h_end, w_start:w_end] += (
                        self.filters[k][None, :, :, :] * d_out[:, k, i, j][:, None, None, None]
                    )

                d_biases += np.sum(d_out[:, :, i, j], axis=0)

        # 파라미터 업데이트 (간단한 SGD)
        self.filters -= lr * d_filters
        self.biases -= lr * d_biases

        # 패딩 제거하고 반환
        if p > 0:
            return d_input_padded[:, :, p:-p, p:-p]
        return d_input_padded


In [11]:
# ReLU 클래스 사용

class ReLU:
    def forward(self, input):
        self.last_input = input
        return np.maximum(0, input)

    def backward(self, d_out):
        return d_out * (self.last_input > 0)

In [24]:
#MaxPool2D 레이어
class MaxPool2D:
    def __init__(self, pool_size=2, stride=2):
        self.pool_size = pool_size
        self.stride = stride

    def forward(self, input):
        self.last_input = input
        batch, channels, h, w = input.shape
        p, s = self.pool_size, self.stride

        out_h = (h - p) // s + 1
        out_w = (w - p) // s + 1

        output = np.zeros((batch, channels, out_h, out_w))
        self.max_indices = {}  # backward에서 어디가 max였는지 기억

        for i in range(out_h):
            for j in range(out_w):
                h_start, h_end = i*s, i*s + p
                w_start, w_end = j*s, j*s + p
                window = input[:, :, h_start:h_end, w_start:w_end]
                output[:, :, i, j] = np.max(window, axis=(2,3))

        return output

    def backward(self, d_out):
        input = self.last_input
        batch, channels, h, w = input.shape
        p, s = self.pool_size, self.stride
        d_input = np.zeros_like(input)

        out_h, out_w = d_out.shape[2], d_out.shape[3]

        for i in range(out_h):
            for j in range(out_w):
                h_start, h_end = i*s, i*s + p
                w_start, w_end = j*s, j*s + p
                window = input[:, :, h_start:h_end, w_start:w_end]

                # 각 채널에서 max였던 위치에만 gradient 전달
                max_vals = np.max(window, axis=(2,3), keepdims=True)
                mask = (window == max_vals)

                d_input[:, :, h_start:h_end, w_start:w_end] += (
                    mask * d_out[:, :, i, j][:, :, None, None]
                )

        return d_input

In [25]:
#Flatten 레이어
class Flatten:
    def forward(self, input):
        self.input_shape = input.shape
        return input.reshape(input.shape[0], -1)

    def backward(self, d_out):
        return d_out.reshape(self.input_shape)

In [26]:
#Dense 레이어
class Dense:
    def __init__(self, input_size, output_size):
        scale = np.sqrt(2.0 / input_size)
        self.weights = np.random.randn(input_size, output_size) * scale
        self.biases = np.zeros(output_size)

    def forward(self, input):
        self.last_input = input
        return input @ self.weights + self.biases

    def backward(self, d_out, lr=0.001):
        d_weights = self.last_input.T @ d_out
        d_biases = np.sum(d_out, axis=0)
        d_input = d_out @ self.weights.T

        batch = self.last_input.shape[0]
        self.weights -= lr * d_weights
        self.biases -= lr * d_biases

        return d_input


In [27]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))  # overflow 방지
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy_loss(probs, labels):

    #probs: softmax 출력 (batch, num_classes)
    #labels: 정수 라벨 (batch,)

    batch = probs.shape[0]
    log_likelihood = -np.log(probs[range(batch), labels] + 1e-9)
    loss = np.sum(log_likelihood) / batch

    # gradient (softmax + cross-entropy의 gradient는 간단히 probs - one_hot)
    d_out = probs.copy()
    d_out[range(batch), labels] -= 1
    d_out /= batch

    return loss, d_out

In [28]:
## 테스트 해보기

if __name__ == "__main__":
    # 더미 데이터로 forward/backward 가 에러 없이 도는지 확인하기
    batch_size = 4
    dummy_input = np.random.randn(batch_size, 3, 32, 32)  # CIFAR-10 사이즈
    dummy_labels = np.random.randint(0, 10, size=batch_size)

    conv1 = Conv2D(num_filters=8, filter_size=3, input_channels=3, padding=1)
    relu1 = ReLU()
    pool1 = MaxPool2D()
    flatten = Flatten()

    # forward
    out = conv1.forward(dummy_input)
    print("Conv1 output shape:", out.shape)  # (4, 8, 32, 32)

    out = relu1.forward(out)
    out = pool1.forward(out)
    print("Pool1 output shape:", out.shape)  # (4, 8, 16, 16)

    out = flatten.forward(out)
    print("Flatten output shape:", out.shape)  # (4, 2048)

    dense = Dense(input_size=out.shape[1], output_size=10)
    logits = dense.forward(out)
    probs = softmax(logits)

    loss, d_logits = cross_entropy_loss(probs, dummy_labels)
    print("Initial loss:", loss)  # 대략 log(10) ≈ 2.3 근처가 정상임 (랜덤 초기화)

    # backward 체인 테스트
    d_out = dense.backward(d_logits)
    d_out = flatten.backward(d_out)
    d_out = pool1.backward(d_out)
    d_out = relu1.backward(d_out)
    d_out = conv1.backward(d_out)

    print("Backward pass 완료, 에러 없음")

Conv1 output shape: (4, 8, 32, 32)
Pool1 output shape: (4, 8, 16, 16)
Flatten output shape: (4, 2048)
Initial loss: 4.277208580215198
Backward pass 완료, 에러 없음


In [30]:
# Keras에 내장된 데이터셋 유틸리티로 다운로드만 사용
# (모델 자체는 Numpy로 직접 구현한 것을 사용함)

from tensorflow.keras.datasets import cifar10

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 정규화 (0~255 -> 0~1)
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Keras는 (N, H, W, C) 순서 -> 우리 구현은 (N, C, H, W) 순서를 쓰므로 변환
x_train = x_train.transpose(0, 3, 1, 2)
x_test = x_test.transpose(0, 3, 1, 2)

y_train = y_train.flatten()
y_test = y_test.flatten()

print("x_train shape:", x_train.shape)  # (50000, 3, 32, 32)
print("x_test shape:", x_test.shape)    # (10000, 3, 32, 32)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

x_train shape: (50000, 3, 32, 32)
x_test shape: (10000, 3, 32, 32)


In [31]:
# 지금 구현한 Conv2D는 반복문(for) 기반이라 매우 느립니다.
# GPU 가속도 없고, 벡터화(im2col 등)도 안 되어있고 50,000장 전체 * 여러 epoch를 돌리면 몇 시간이 걸릴 수도 있음.
# 그래서 여기서는 "전체 CIFAR-10"이 아니라 "작은 서브셋"으로 학습을 진행. (원리 검증 + 시간 절약 목적)

SUBSET_TRAIN = 1000  # 학습에 쓸 이미지 수
SUBSET_TEST = 500

np.random.seed(42)
train_idx = np.random.choice(len(x_train), SUBSET_TRAIN, replace=False)
test_idx = np.random.choice(len(x_test), SUBSET_TEST, replace=False)

x_train_small = x_train[train_idx]
y_train_small = y_train[train_idx]
x_test_small = x_test[test_idx]
y_test_small = y_test[test_idx]

print(f"학습에 사용할 서브셋: {SUBSET_TRAIN}장, 테스트: {SUBSET_TEST}장")

학습에 사용할 서브셋: 1000장, 테스트: 500장


In [32]:
# 모델 조립 (SimpleCNN 클래스)

class SimpleCNN:
    def __init__(self, num_classes=10, lr=0.01):
        self.lr = lr

        # 먼저 설계한 아키텍처를 그대로 씀
        self.conv1 = Conv2D(num_filters=8, filter_size=3, input_channels=3, padding=1)
        self.relu1 = ReLU()
        self.pool1 = MaxPool2D()  # 32x32 -> 16x16

        self.conv2 = Conv2D(num_filters=16, filter_size=3, input_channels=8, padding=1)
        self.relu2 = ReLU()
        self.pool2 = MaxPool2D()  # 16x16 -> 8x8

        self.flatten = Flatten()
        self.dense1 = Dense(input_size=16 * 8 * 8, output_size=64)
        self.relu3 = ReLU()
        self.dense2 = Dense(input_size=64, output_size=num_classes)

        # 참고: 원래 설계한 3-Conv 구조보다 필터 수를 줄인 경량 버전임.
        # (NumPy 반복문 구현이 느려서 실습용으로 축소.

    def forward(self, x):
        x = self.conv1.forward(x)
        x = self.relu1.forward(x)
        x = self.pool1.forward(x)

        x = self.conv2.forward(x)
        x = self.relu2.forward(x)
        x = self.pool2.forward(x)

        x = self.flatten.forward(x)
        x = self.dense1.forward(x)
        x = self.relu3.forward(x)
        x = self.dense2.forward(x)

        return x

    def backward(self, d_out):
        d_out = self.dense2.backward(d_out, lr=self.lr)
        d_out = self.relu3.backward(d_out)
        d_out = self.dense1.backward(d_out, lr=self.lr)

        d_out = self.flatten.backward(d_out)

        d_out = self.pool2.backward(d_out)
        d_out = self.relu2.backward(d_out)
        d_out = self.conv2.backward(d_out, lr=self.lr)

        d_out = self.pool1.backward(d_out)
        d_out = self.relu1.backward(d_out)
        d_out = self.conv1.backward(d_out, lr=self.lr)

In [33]:
# 학습 루프
def train(model, x_train, y_train, x_val, y_val, epochs=5, batch_size=32):
    num_samples = x_train.shape[0]
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        start = time.time()

        # 매 epoch마다 셔플
        perm = np.random.permutation(num_samples)
        x_shuffled = x_train[perm]
        y_shuffled = y_train[perm]

        epoch_loss = 0.0
        correct = 0
        num_batches = num_samples // batch_size

        for b in range(num_batches):
            x_batch = x_shuffled[b*batch_size:(b+1)*batch_size]
            y_batch = y_shuffled[b*batch_size:(b+1)*batch_size]

            # forward
            logits = model.forward(x_batch)
            probs = softmax(logits)
            loss, d_logits = cross_entropy_loss(probs, y_batch)

            # backward + 파라미터 업데이트
            model.backward(d_logits)

            epoch_loss += loss
            correct += np.sum(np.argmax(probs, axis=1) == y_batch)

            if b % 20 == 0:
                print(f"  epoch {epoch+1} batch {b}/{num_batches} loss={loss:.4f}")

        train_acc = correct / (num_batches * batch_size)
        avg_loss = epoch_loss / num_batches

        # 검증
        val_logits = model.forward(x_val)
        val_probs = softmax(val_logits)
        val_acc = np.mean(np.argmax(val_probs, axis=1) == y_val)

        elapsed = time.time() - start
        print(f"[Epoch {epoch+1}/{epochs}] loss={avg_loss:.4f} "
              f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} "
              f"({elapsed:.1f}s)")

        history["train_loss"].append(avg_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

    return history


In [34]:
model = SimpleCNN(num_classes=10, lr=0.01)

history = train(
    model,
    x_train_small, y_train_small,
    x_test_small, y_test_small,
    epochs=5,
    batch_size=32,
)

print("\n최종 결과:", history)

  epoch 1 batch 0/31 loss=2.7646
  epoch 1 batch 20/31 loss=2.2525
[Epoch 1/5] loss=2.4159 train_acc=0.1058 val_acc=0.1320 (34.3s)
  epoch 2 batch 0/31 loss=2.1887
  epoch 2 batch 20/31 loss=2.1655
[Epoch 2/5] loss=2.2603 train_acc=0.1704 val_acc=0.1480 (31.6s)
  epoch 3 batch 0/31 loss=2.1957
  epoch 3 batch 20/31 loss=2.1972
[Epoch 3/5] loss=2.1890 train_acc=0.1925 val_acc=0.1680 (31.0s)
  epoch 4 batch 0/31 loss=2.1181
  epoch 4 batch 20/31 loss=2.1828
[Epoch 4/5] loss=2.1549 train_acc=0.2177 val_acc=0.1640 (32.5s)
  epoch 5 batch 0/31 loss=2.1953
  epoch 5 batch 20/31 loss=2.1235
[Epoch 5/5] loss=2.1073 train_acc=0.2450 val_acc=0.2240 (31.2s)

최종 결과: {'train_loss': [np.float64(2.4158733987886167), np.float64(2.260270513321995), np.float64(2.1889583350327704), np.float64(2.154901768662892), np.float64(2.107340869026519)], 'train_acc': [np.float64(0.10584677419354839), np.float64(0.17036290322580644), np.float64(0.19254032258064516), np.float64(0.21774193548387097), np.float64(0.2449